# Sports Vision: training en evaluatie
Andrew, Milan, Raoul en Jorre - 14 november 2026.

Dit notebook is een achteraf samengesteld, uitvoerbaar overzicht. De historische resultaten komen uit de meegeleverde checkpoints en originele notebooks. De huidige overzichtscellen zijn echt uitgevoerd bij het verpakken. Ze bootsen geen oorspronkelijke GPU-training na. Nieuwe training staat standaard uit.

In [1]:
from pathlib import Path
import json, collections, hashlib
ROOT = Path.cwd()
if not (ROOT/'pyproject.toml').exists():
    raise RuntimeError('Open dit notebook met 03_Code als werkmap.')
RUN_TRAINING = False
DEVICE = 'cpu'  # Kies expliciet een beschikbare CUDA-device voor GPU-training.
meta=json.loads((ROOT/'reports/submission/checkpoint_metadata.json').read_text(encoding='utf-8'))
print('Bronversie: main 524d081a; nieuwe training:', RUN_TRAINING)

Bronversie: main 524d081a; nieuwe training: False


## 1. Definitieve labels en afzonderlijke trainingsdata
De reviewselectie van 80 beelden is een eigen controlebijlage. De eerder getrainde checkpoints zijn niet achteraf opnieuw getraind op deze reviewregistratie.

In [2]:
for name in ['datasets/school-ball80-review-v1/manifest.json','data/broadcast-multisource-v1/manifest.json']:
    d=json.loads((ROOT/name).read_text(encoding='utf-8'))
    print(name,dict(collections.Counter(r['split'] for r in d['frames'])))
d=json.loads((ROOT/'datasets/school-ball80-review-v1/manifest.json').read_text(encoding='utf-8'))
print('Menselijk gecontroleerd:',d['human_review_completed'])
print('Reviewstatus:',dict(collections.Counter(r['review_status'] for r in d['frames'])))

datasets/school-ball80-review-v1/manifest.json {'train': 48, 'val': 32}
data/broadcast-multisource-v1/manifest.json {'train': 3936, 'val': 348}
Menselijk gecontroleerd: True
Reviewstatus: {'approved': 80}


## 2. Gekozen modellen en echte opgeslagen metrics
Dit zijn frameworkmetrics uit de checkpointmetadata. De spelers- en Andrew-resultaten gebruiken verschillende datasets en zijn geen onderlinge rangschikking. De bredere pipeline-evaluatie staat afzonderlijk in de rapporten.

In [3]:
for key in ['players','andrew']:
    m=meta[key]
    print(key, m['yaml_file'],m['scale'],m['names'])
    print('training:',{k:m['train_args'].get(k) for k in ['model','epochs','imgsz','batch','data']})
    print('metrics:',m['train_metrics'])
    assert hashlib.sha256((ROOT/m['weights']).read_bytes()).hexdigest()==m['sha256']

players yolo11s.yaml s {'0': 'player'}
training: {'model': 'C:\\Messi10\\artifacts\\broadcast-refine-v2\\detector\\batch-2\\weights\\best.pt', 'epochs': 40, 'imgsz': 1536, 'batch': 2, 'data': 'C:\\Messi10\\data\\broadcast-multisource-v1\\data.yaml'}
metrics: {'metrics/precision(B)': 0.90481, 'metrics/recall(B)': 0.86982, 'metrics/mAP50(B)': 0.93609, 'metrics/mAP50-95(B)': 0.60275, 'val/box_loss': 1.21679, 'val/cls_loss': 0.68482, 'val/dfl_loss': 1.00145, 'fitness': 0.60275}
andrew None None {'0': 'ball', '1': 'goalkeeper', '2': 'player', '3': 'referee'}
training: {'model': 'yolov8x.pt', 'epochs': 50, 'imgsz': 1280, 'batch': 1, 'data': 'archive\\data.yaml'}
metrics: {'metrics/precision(B)': 0.94554, 'metrics/recall(B)': 0.8471, 'metrics/mAP50(B)': 0.88529, 'metrics/mAP50-95(B)': 0.66698, 'val/box_loss': 0.66758, 'val/cls_loss': 0.32608, 'val/dfl_loss': 0.81437, 'fitness': 0.66698}


## 3. Originele bijdrage van Andrew
Lees Andrew_Prototype/train.ipynb en check_mAP.ipynb. De broncode start met yolov8x.pt. De vier uitvoerklassen zijn ball, goalkeeper, player en referee. De gezamenlijke app gebruikt uit dit model alleen ball. De originele notebookcode vraagt batch 6; de aangeleverde checkpointmetadata vermeldt batch 1. Beide blijven als historische bron bewaard.

In [4]:
nb=json.loads((ROOT/'Andrew_Prototype/train.ipynb').read_text(encoding='utf-8'))
for cell in nb['cells']:
    source=''.join(cell.get('source',[]))
    if 'model.train(' in source:
        print(source)


import torch
from ultralytics import YOLO

#GPU CHECK

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Check your PyTorch installation.")

print("GPU:", torch.cuda.get_device_name(0))
print("Total GPU memory:", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
), "GB")

#LIMIT PYTORCH GPU MEM TO 85%

torch.cuda.set_per_process_memory_fraction(0.85, 0)

print("PyTorch GPU memory limit: 85%")

#TRAINING

model = YOLO("yolov8x.pt")

results = model.train(
    data="archive/data.yaml",
    
    epochs=50,
    imgsz=1280,
    batch=6,

    patience=20,
    workers=2,
    device=0,

    project="soccer_training",
    name="ball_detector_640",

    save=True,
    plots=True,
    amp=True
)

print("\nTraining completed!")

print("Best model saved at:")
print("soccer_training/soccer_yolo/weights/best.pt")

print("Last checkpoint saved at:")
print("soccer_training/soccer_yolo/weights/last.pt")



## 4. Lokale trainingsbestanden
De oorspronkelijke manifests behouden hun bronpaden. Voor een nieuwe training maken we een nieuwe lokale data.yaml. De spelersdata bevatten 3.936 train- en 348 validatiebeelden. Andrews data bevatten 612 train-, 38 validatie- en 13 testbeelden.

In [5]:
def local_yaml(kind):
    folder=ROOT/'.cache/notebook-data'
    folder.mkdir(parents=True,exist_ok=True)
    if kind=='players':
        base=ROOT/'data/broadcast-multisource-v1'
        text=f'path: {base.as_posix()}\ntrain: images/train\nval: images/val\nnames:\n  0: player\n'
    else:
        base=ROOT/'Andrew_Prototype/archive'
        text=f'path: {base.as_posix()}\ntrain: train/images\nval: valid/images\ntest: test/images\nnames:\n  0: ball\n  1: goalkeeper\n  2: player\n  3: referee\n'
    output=folder/(kind+'.yaml')
    output.write_text(text,encoding='utf-8')
    return output
for kind in ['players','andrew']:
    print(kind,local_yaml(kind).read_text(encoding='utf-8'))

players path: C:/CV/Messi10/Upload_CV_Challenge_2026_11_14/03_Code/data/broadcast-multisource-v1
train: images/train
val: images/val
names:
  0: player

andrew path: C:/CV/Messi10/Upload_CV_Challenge_2026_11_14/03_Code/Andrew_Prototype/archive
train: train/images
val: valid/images
test: test/images
names:
  0: ball
  1: goalkeeper
  2: player
  3: referee



## 5. Optionele nieuwe YOLO11s-training
De exacte eerdere tusseninitialisatie is niet meegeleverd. Onderstaande cel begint daarom bij pretrained YOLO11s en schrijft een nieuwe run. Dit demonstreert dezelfde architectuur en dataset, maar reproduceert niet exact de volledige opeenvolging van historische tussenruns. De oorspronkelijke trainingsorkestratie staat in src/football_pipeline/broadcast_refine.py en broadcast_release.py.

In [6]:
if RUN_TRAINING:
    from ultralytics import YOLO
    model=YOLO(str(ROOT/'models/yolo11s.pt'))
    args=meta['players']['train_args']
    result=model.train(data=str(local_yaml('players')),epochs=40,imgsz=1536,batch=2,
                       optimizer=args.get('optimizer','AdamW'),lr0=args.get('lr0',0.0001),
                       device=DEVICE,project=str(ROOT/'runs/notebook-training'),name='players-new',exist_ok=False)
else:
    print('Nieuwe spelers-training uitgeschakeld. Bekijk de opgeslagen checkpointmetadata en originele scripts.')

Nieuwe spelers-training uitgeschakeld. Bekijk de opgeslagen checkpointmetadata en originele scripts.


## 6. Optionele nieuwe training van Andrews balmodel
De originele vierklassentaak blijft behouden. Alleen ball trainen zou een andere taak en een nieuwe vergelijkingsmeting vereisen. Een pretrained yolov8x-checkpoint downloadt Ultralytics bij deze expliciete nieuwe training indien nodig.

In [7]:
if RUN_TRAINING:
    from ultralytics import YOLO
    model=YOLO('yolov8x.pt')
    model.train(data=str(local_yaml('andrew')),epochs=50,imgsz=1280,batch=1,
                device=DEVICE,project=str(ROOT/'runs/notebook-training'),name='andrew-new',exist_ok=False)
else:
    print('Nieuwe Andrew-training uitgeschakeld. De originele uitvoer staat in Andrew_Prototype/train.ipynb.')

Nieuwe Andrew-training uitgeschakeld. De originele uitvoer staat in Andrew_Prototype/train.ipynb.


## 7. Validatie en foutanalyse
Gebruik voor een nieuwe run model.val(data=..., split="val"). Rapporteer mAP50 en mAP50-95 naast de gebruikte bron, klasse en confidence. Gebruik test niet voor modelselectie. Voorbeelden en hypotheses staan in docs/Resultaten_Balexperimenten.md, docs/Broadcast_Betrouwbaarheid_20261030.md en het PDF-verslag. De brede developmentset is herhaald gebruikt voor keuzes en vormt geen onafhankelijke eindtest.

In [8]:
checks=json.loads((ROOT/'reports/release-20261114/checks.json').read_text(encoding='utf-8'))
print('Softwaretests:',checks['tests'])
print('Geen nieuwe training of onafhankelijke eindtest uitgevoerd bij het verpakken.')

Softwaretests: {'passed': 537, 'skipped': 1, 'subtests_passed': 4, 'failures': 0, 'errors': 0, 'runtime_s': 79.431, 'skipped_reason': 'GPU smoke test: no CUDA device in this environment'}
Geen nieuwe training of onafhankelijke eindtest uitgevoerd bij het verpakken.
